# Hotel Booking Analysis

Loading 20 sample hotel bookings with the `HotelBooking` class, then
exploring them with plain Python and pandas.

1. Import and Load Data

In [1]:
# Import my classes
from booking import Booking, HotelBooking

In [2]:
# CSV Load function created

import csv

def load_bookings(path):
    with open(path, newline ="") as f:
        reader = csv.DictReader(f)
        bookings = []
        for row in reader:
            bookings.append(HotelBooking.from_dict(row))
        return bookings    
    
bookings = load_bookings("bookings.csv")


In [3]:
# Quick check if data is good.

print(len(bookings))
print(bookings[0])
print(Booking.booking_count)
print(HotelBooking.booking_count)

b = bookings[0]
print(type(b.price),b.price)
print(type(b.start_date), b.start_date)
print("Booking class commission rate: ", Booking.commission_rate)
print("HotelBooking class commission rate: ", HotelBooking.commission_rate)

20
HotelBooking(booking_id=1001, guest_name='KukHwa Kim', hotel_name='Grand Palace Hotel', price=1250.0, status='confirmed')
20
20
<class 'float'> 1250.0
<class 'datetime.date'> 2026-10-15
Booking class commission rate:  0.05
HotelBooking class commission rate:  0.111


2. Plain Python functions: filter, sort, sum, average, group by, list comprehension

In [4]:
# Filtering

def filter_by_status(bookings, status):
    filtered = []
    for booking in bookings:
        if booking.status == status:
            filtered.append(booking)
    return filtered

confirmed = filter_by_status(bookings, "confirmed")
print(len(confirmed))
for b in confirmed:
    print(b.booking_id, b.status)

cancelled = filter_by_status(bookings, "cancelled")
print(len(cancelled))
for b in cancelled:
    print(b.booking_id, b.status)    

14
1001 confirmed
1003 confirmed
1004 confirmed
1005 confirmed
1007 confirmed
1008 confirmed
1010 confirmed
1012 confirmed
1013 confirmed
1014 confirmed
1016 confirmed
1018 confirmed
1019 confirmed
1020 confirmed
3
1006 cancelled
1011 cancelled
1017 cancelled


In [5]:
# list comprehension

def filter_by_status(bookings, status):
    return [b for b in bookings if b.status == status]

print(len(filter_by_status(bookings,"confirmed")))
print(len(filter_by_status(bookings,"cancelled")))
print([x.booking_id for x in filter_by_status(bookings,"confirmed")])

def filter_by_provider(bookings,provider):
    return [b for b in bookings if b.provider == provider]

print(len(filter_by_provider(bookings,"Expedia")))
print(len(filter_by_provider(bookings,"Booking.com")))


14
3
[1001, 1003, 1004, 1005, 1007, 1008, 1010, 1012, 1013, 1014, 1016, 1018, 1019, 1020]
5
8


In [6]:
# sort
def sort_by_price(bookings,descending=False):
    return sorted(bookings, key=lambda b:b.price, reverse=descending)

print([(x.booking_id,x.price) for x in sort_by_price(bookings,descending=False)[:3]]) # smallest one first
print([(x.booking_id,x.price) for x in sort_by_price(bookings,descending=True)[:3]]) # biggest one first

def sort_by_checkin(bookings,descending=False):
    return sorted(bookings, key=lambda b: b.start_date, reverse=descending) 

print([(x.booking_id, x.start_date) for x in sort_by_checkin(bookings,descending=False)[:3]]) #smallest one first
print([(x.booking_id, x.start_date) for x in sort_by_checkin(bookings,descending=True)[:3]])  #biggest one first

[(1015, 380.0), (1018, 390.0), (1008, 420.0)]
[(1013, 4200.0), (1019, 2980.0), (1007, 2650.0)]
[(1005, datetime.date(2026, 10, 2)), (1015, datetime.date(2026, 10, 2)), (1006, datetime.date(2026, 10, 8))]
[(1016, datetime.date(2027, 1, 10)), (1013, datetime.date(2026, 12, 28)), (1019, datetime.date(2026, 12, 22))]


In [7]:
# Sum and Average
def total_revenue(bookings):
    return sum( b.price for b in  bookings)

def average_price(bookings):
    if not bookings:
        return 0
    else:
        return round(total_revenue(bookings)/len(bookings),2)

def total_commission(bookings):
    return round(sum(b.commission() for b in bookings),2)

print(average_price(bookings))
print(average_price([]))
print(total_commission(bookings))


1393.81
0
2128.13


In [8]:
# Group by
def count_by_provider(bookings):
     counts = {}
     for b in bookings:
          if b.provider not in counts:
               counts[b.provider] = 1
          else:
               counts[b.provider] += 1 

     return counts
print(count_by_provider(bookings))


def revenue_by_provider(bookings):
    sum_revenue={}
    for b in bookings:
         if b.provider not in sum_revenue:
              sum_revenue[b.provider] = b.price 
         else:
              sum_revenue[b.provider] += b.price 
    return sum_revenue

print(revenue_by_provider(bookings))




{'Booking.com': 8, 'Agoda': 4, 'Expedia': 5, 'Hotels.com': 3}
{'Booking.com': 9370.75, 'Agoda': 6490.5, 'Expedia': 7829.99, 'Hotels.com': 4185.0}


3. pandas: the same analysis, faster

In [9]:
# import pandas

import pandas as pd 

df = pd.read_csv("bookings.csv", parse_dates=["booking_date","start_date","end_date"])

df.head(5)

,booking_id,guest_name,hotel_name,room_type,booking_date,start_date,end_date,depart_country,destination_country,price,provider,status,commission_rate
0,1001,KukHwa Kim,Grand Palace Hotel,Double,2026-09-01,2026-10-15,2026-10-25,Canada,Spain,1250.00,Booking.com,confirmed,NaN
1,1002,Sarang Kang,Seaside Resort,Suite,2026-09-10,2026-11-05,2026-11-10,South Korea,Thailand,980.50,Agoda,pending,NaN
2,1003,John Smith,Seaside Resort,Double,2026-09-11,2026-11-05,2026-11-10,United States,Thailand,999.99,Expedia,confirmed,0.03
3,1004,Minji Park,Hotel Le Germain,King,2026-09-15,2026-12-20,2026-12-27,South Korea,Canada,2140.75,Booking.com,confirmed,0.06
4,1005,Lucas Tremblay,Ocean View Inn,Queen,2026-09-20,2026-10-02,2026-10-05,Canada,Portugal,615.00,Hotels.com,confirmed,NaN


In [10]:
# filter_by_status
print([b.booking_id for b in filter_by_status(bookings,"confirmed")])

print(df[df["status"]=="confirmed"]["booking_id"].to_list())


[1001, 1003, 1004, 1005, 1007, 1008, 1010, 1012, 1013, 1014, 1016, 1018, 1019, 1020]
[1001, 1003, 1004, 1005, 1007, 1008, 1010, 1012, 1013, 1014, 1016, 1018, 1019, 1020]


In [11]:
# sort by price
print([(b.booking_id,b.price) for b in sort_by_price(bookings,descending=True)[:3]])

print(list(df.sort_values("price", ascending=False)[:3][["booking_id","price"]].itertuples(index=False,name=None)))

[(1013, 4200.0), (1019, 2980.0), (1007, 2650.0)]
[(1013, 4200.0), (1019, 2980.0), (1007, 2650.0)]


In [12]:
#total revenue & average price
print("total revenue:", total_revenue(bookings))
print("average price:", average_price(bookings))

print(df["price"].sum())
print(df["price"].mean())

assert total_revenue(bookings) == df["price"].sum() # faster comparision 

total revenue: 27876.239999999998
average price: 1393.81
27876.239999999998
1393.812


In [13]:
# count by provider & revenue by provider
print(count_by_provider(bookings))
print(revenue_by_provider(bookings))

print(df.groupby("provider",sort=False).size().to_dict())
print(df.groupby("provider",sort=False)["price"].sum().to_dict())

{'Booking.com': 8, 'Agoda': 4, 'Expedia': 5, 'Hotels.com': 3}
{'Booking.com': 9370.75, 'Agoda': 6490.5, 'Expedia': 7829.99, 'Hotels.com': 4185.0}
{'Booking.com': 8, 'Agoda': 4, 'Expedia': 5, 'Hotels.com': 3}
{'Booking.com': 9370.75, 'Agoda': 6490.5, 'Expedia': 7829.99, 'Hotels.com': 4185.0}
